# Bottle Scan, Align, and Depth Approach Demo

Real Jetson/JETANK integration test using the proven `fcn-mobilenet` DepthNet route and the built-in UFF `ssd-mobilenet-v2` detector.

The workflow is: initialize models and camera, move the arm to a tunable camera-safe pose, run a continuous-capture Search State, align the bottle bbox center, run a stationary-base Approaching Arm State, realign, then approach in bounded pulses until the bottle-centered raw DepthNet mean is below `stop_depth_m`. DepthNet values are relative model outputs, not calibrated meters.

In [ ]:
import json
import os
import sys
import threading
import time
from pathlib import Path

import cv2
import ipywidgets as widgets
import numpy as np
from IPython.display import display

JETSON_INFERENCE_ROOT = Path('/workspace/jetson-inference')
JETSON_INFERENCE_DATA = JETSON_INFERENCE_ROOT / 'data'
SSD_MODEL_DIR = JETSON_INFERENCE_DATA / 'networks' / 'SSD-Mobilenet-v2'
SSD_MODEL_PATH = SSD_MODEL_DIR / 'ssd_mobilenet_v2_coco.uff'
SSD_LABELS_PATH = SSD_MODEL_DIR / 'ssd_coco_labels.txt'
MODEL_MANIFEST_PATH = JETSON_INFERENCE_DATA / 'networks' / 'models.json'
DEPTH_MODEL_PATH = JETSON_INFERENCE_DATA / 'networks' / 'MonoDepth-FCN-Mobilenet' / 'monodepth_fcn_mobilenet.onnx'
DEPTH_NETWORK_NAME = 'fcn-mobilenet'
DETECT_NETWORK_NAME = 'ssd-mobilenet-v2'
CAMERA_WIDTH = 320
CAMERA_HEIGHT = 240
PARAM_PATH = Path('bottle_approach_params.json')

for path in [
    JETSON_INFERENCE_ROOT / 'build/aarch64/lib/python/3.6',
    JETSON_INFERENCE_ROOT / 'python/examples',
]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
library_path = JETSON_INFERENCE_ROOT / 'build/aarch64/lib'
os.environ['LD_LIBRARY_PATH'] = str(library_path) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

DEFAULT_PARAMS = {
    'dry_run_base': True,
    'dry_run_arm': True,
    'detection_confidence_k': 0.35,
    'stop_depth_m': 1.60,
    'depth_roi_half_size_norm': 0.06,
    'camera_x_sign': 1,
    'scan_direction': 'right',
    'scan_speed': 0.15,
    'scan_step_seconds': 0.15,
    'scan_full_turn_seconds': 4.0,
    'align_turn_speed': 0.14,
    'align_pulse_seconds': 0.08,
    'align_tolerance_norm': 0.08,
    'align_max_steps': 30,
    'approach_speed': 0.20,
    'approach_pulse_seconds': 0.15,
    'approach_max_seconds': 10.0,
    'observation_pause_seconds': 0.10,
    'safe_s1': 0, 'safe_s2': 0, 'safe_s3': 0, 'safe_s4': 0, 'safe_s5': 0,
    'safe_order': [5, 4, 3, 2, 1],
    'arm_speed': 85,
    'servo_settle_seconds': 0.35,
    'search_confidence_k': 0.30,
    'search_turn_speed': 0.15,
    'search_max_seconds': 4.0,
    'approaching_s1': 0, 'approaching_s2': 0, 'approaching_s3': 0, 'approaching_s4': 0, 'approaching_s5': 0,
}


def load_params():
    loaded = dict(DEFAULT_PARAMS)
    if PARAM_PATH.exists():
        saved = json.loads(PARAM_PATH.read_text())
        loaded.update(saved)
        if 'search_confidence_k' not in saved:
            loaded['search_confidence_k'] = float(saved.get('detection_confidence_k', DEFAULT_PARAMS['search_confidence_k']))
        if 'search_turn_speed' not in saved:
            loaded['search_turn_speed'] = float(saved.get('scan_speed', DEFAULT_PARAMS['search_turn_speed']))
        if 'search_max_seconds' not in saved:
            loaded['search_max_seconds'] = float(saved.get('scan_full_turn_seconds', DEFAULT_PARAMS['search_max_seconds']))
        for servo_id in range(1, 6):
            name = 'approaching_s{}'.format(servo_id)
            if name not in saved:
                loaded[name] = int(saved.get('safe_s{}'.format(servo_id), 0))
    return loaded


params = load_params()
print('[params] loaded from', PARAM_PATH if PARAM_PATH.exists() else 'defaults')

In [ ]:
def float_slider(name, minimum, maximum, step, width='540px'):
    return widgets.FloatSlider(
        value=float(params[name]), min=minimum, max=maximum, step=step,
        description=name, continuous_update=False, readout_format='.2f',
        style={'description_width': '190px'}, layout=widgets.Layout(width=width)
    )


def int_slider(name, minimum, maximum, step=1, width='540px'):
    return widgets.IntSlider(
        value=int(params[name]), min=minimum, max=maximum, step=step,
        description=name, continuous_update=False,
        style={'description_width': '190px'}, layout=widgets.Layout(width=width)
    )


def angle_slider(name):
    return int_slider(name, -180, 180)


controls = {
    'dry_run_base': widgets.Checkbox(value=bool(params['dry_run_base']), description='dry_run_base'),
    'dry_run_arm': widgets.Checkbox(value=bool(params['dry_run_arm']), description='dry_run_arm'),
    'detection_confidence_k': float_slider('detection_confidence_k', 0.05, 0.95, 0.05),
    'stop_depth_m': float_slider('stop_depth_m', 0.10, 5.00, 0.05),
    'depth_roi_half_size_norm': float_slider('depth_roi_half_size_norm', 0.01, 0.20, 0.01),
    'camera_x_sign': widgets.Dropdown(
        value=int(params['camera_x_sign']), options=[('normal', 1), ('mirrored', -1)],
        description='camera_x_sign', style={'description_width': '190px'}, layout=widgets.Layout(width='390px')
    ),
    'scan_direction': widgets.Dropdown(
        value=params['scan_direction'], options=['left', 'right'],
        description='scan_direction', style={'description_width': '190px'}, layout=widgets.Layout(width='390px')
    ),
    'scan_speed': float_slider('scan_speed', 0.0, 0.5, 0.01),
    'scan_step_seconds': float_slider('scan_step_seconds', 0.05, 0.50, 0.01),
    'scan_full_turn_seconds': float_slider('scan_full_turn_seconds', 0.5, 15.0, 0.1),
    'align_turn_speed': float_slider('align_turn_speed', 0.0, 0.5, 0.01),
    'align_pulse_seconds': float_slider('align_pulse_seconds', 0.02, 0.50, 0.01),
    'align_tolerance_norm': float_slider('align_tolerance_norm', 0.01, 0.40, 0.01),
    'align_max_steps': int_slider('align_max_steps', 1, 100),
    'approach_speed': float_slider('approach_speed', 0.0, 0.5, 0.01),
    'approach_pulse_seconds': float_slider('approach_pulse_seconds', 0.05, 0.50, 0.01),
    'approach_max_seconds': float_slider('approach_max_seconds', 0.5, 30.0, 0.5),
    'observation_pause_seconds': float_slider('observation_pause_seconds', 0.0, 1.0, 0.05),
    'search_confidence_k': float_slider('search_confidence_k', 0.05, 0.95, 0.05),
    'search_turn_speed': float_slider('search_turn_speed', 0.0, 0.5, 0.01),
    'search_max_seconds': float_slider('search_max_seconds', 0.5, 15.0, 0.1),
    'safe_s1': angle_slider('safe_s1'),
    'safe_s2': angle_slider('safe_s2'),
    'safe_s3': angle_slider('safe_s3'),
    'safe_s4': angle_slider('safe_s4'),
    'safe_s5': angle_slider('safe_s5'),
    'approaching_s1': angle_slider('approaching_s1'),
    'approaching_s2': angle_slider('approaching_s2'),
    'approaching_s3': angle_slider('approaching_s3'),
    'approaching_s4': angle_slider('approaching_s4'),
    'approaching_s5': angle_slider('approaching_s5'),
    'arm_speed': int_slider('arm_speed', 20, 300, 5),
    'servo_settle_seconds': float_slider('servo_settle_seconds', 0.05, 1.5, 0.05),
}
safe_order_widget = widgets.Text(
    value=','.join(str(item) for item in params['safe_order']),
    description='safe_order', placeholder='5,4,3,2,1',
    style={'description_width': '190px'}, layout=widgets.Layout(width='540px')
)


def parse_safe_order(value):
    cleaned = str(value).replace('(', '').replace(')', '').replace('[', '').replace(']', '')
    order = [int(part.strip()) for part in cleaned.split(',') if part.strip()]
    if len(order) != 5 or sorted(order) != [1, 2, 3, 4, 5]:
        raise ValueError('safe_order must contain servo IDs 1-5 exactly once')
    return order


def current_params():
    data = {name: control.value for name, control in controls.items()}
    data['camera_x_sign'] = int(data['camera_x_sign'])
    data['safe_order'] = parse_safe_order(safe_order_widget.value)
    return data


def save_params():
    data = current_params()
    PARAM_PATH.write_text(json.dumps(data, indent=2) + '\n')
    print('[params] saved to', PARAM_PATH)

In [ ]:
depth_net = None
depth_array = None
detector = None
labels = []
bottle_class_ids = []
camera = None
cuda_from_numpy = None
cuda_sync = None
robot = None
ttl_servo = None
base_stop_lock = threading.Lock()


def require_model_files():
    required = [MODEL_MANIFEST_PATH, SSD_MODEL_PATH, SSD_LABELS_PATH, DEPTH_MODEL_PATH]
    missing = [path for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError('missing model files: {}'.format(', '.join(str(path) for path in missing)))


def load_models():
    global depth_net, depth_array, detector, labels, bottle_class_ids, cuda_from_numpy, cuda_sync
    require_model_files()
    from jetson_inference import depthNet, detectNet
    from jetson_utils import cudaDeviceSynchronize, cudaFromNumpy, cudaToNumpy

    cuda_from_numpy = cudaFromNumpy
    cuda_sync = cudaDeviceSynchronize
    if depth_net is None:
        started = time.time()
        print('[model] loading DepthNet {}'.format(DEPTH_NETWORK_NAME))
        depth_net = depthNet(DEPTH_NETWORK_NAME)
        depth_field = depth_net.GetDepthField()
        depth_array = cudaToNumpy(depth_field)
        print('[model] DepthNet loaded in {:.2f}s field={}x{} format={}'.format(
            time.time() - started, depth_field.width, depth_field.height, depth_field.format
        ))

    if detector is None:
        started = time.time()
        previous_cwd = Path.cwd()
        print('[model] loading detector {}'.format(DETECT_NETWORK_NAME))
        try:
            os.chdir(str(JETSON_INFERENCE_DATA))
            detector = detectNet(DETECT_NETWORK_NAME, threshold=float(controls['detection_confidence_k'].value))
        finally:
            os.chdir(str(previous_cwd))
        print('[model] detector loaded in {:.2f}s'.format(time.time() - started))
    else:
        detector.SetThreshold(float(controls['detection_confidence_k'].value))

    labels = [line.strip() for line in SSD_LABELS_PATH.read_text().splitlines() if line.strip()]
    bottle_class_ids = [index for index, label in enumerate(labels) if label.lower() == 'bottle']
    print('[model] labels={} bottle_class_ids={} confidence_k={:.2f}'.format(
        len(labels), bottle_class_ids, float(controls['detection_confidence_k'].value)
    ))
    return depth_net, detector


def start_camera():
    global camera
    if camera is not None:
        print('[camera] already started')
        return camera
    from jetbot import Camera
    print('[camera] starting JetBot Camera {}x{}'.format(CAMERA_WIDTH, CAMERA_HEIGHT))
    try:
        camera = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
        time.sleep(1.0)
        if camera.value is None:
            raise RuntimeError('camera returned no frame')
        print('[camera] connected frame_shape={} dtype={}'.format(camera.value.shape, camera.value.dtype))
    except Exception:
        stop_camera()
        raise
    return camera


def stop_camera():
    global camera
    if camera is None:
        print('[camera] already stopped')
        return
    try:
        camera.stop()
        time.sleep(0.5)
    finally:
        camera = None
    print('[camera] camera.stop complete')


def ensure_robot():
    global robot
    if controls['dry_run_base'].value:
        return None
    if robot is None:
        from jetbot import Robot
        robot = Robot()
        print('[base] Robot connected')
    return robot


def base_stop(reason='manual'):
    with base_stop_lock:
        if robot is not None:
            robot.stop()
    print('[base] stop reason={}'.format(reason))


def start_continuous_search_turn(direction, speed, max_seconds):
    bot = ensure_robot()
    print('[base] continuous search direction={} speed={:.2f} max_seconds={:.2f} dry_run={}'.format(
        direction, float(speed), float(max_seconds), controls['dry_run_base'].value
    ))
    if bot is None:
        return None
    if direction == 'left':
        bot.left(float(speed))
    elif direction == 'right':
        bot.right(float(speed))
    else:
        raise ValueError('unsupported search direction: {}'.format(direction))
    timer = threading.Timer(float(max_seconds), lambda: base_stop('Search maximum time reached'))
    timer.daemon = True
    timer.start()
    return timer


def base_pulse(direction, speed, seconds, label):
    bot = ensure_robot()
    print('[base] {} direction={} speed={:.2f} seconds={:.2f} dry_run={}'.format(
        label, direction, float(speed), float(seconds), controls['dry_run_base'].value
    ))
    if bot is None:
        time.sleep(float(seconds))
        return
    try:
        if direction == 'forward':
            bot.forward(float(speed))
        elif direction == 'left':
            bot.left(float(speed))
        elif direction == 'right':
            bot.right(float(speed))
        else:
            raise ValueError('unsupported direction: {}'.format(direction))
        time.sleep(float(seconds))
    finally:
        bot.stop()


def ensure_servos():
    global ttl_servo
    if controls['dry_run_arm'].value:
        return None
    if ttl_servo is None:
        from SCSCtrl import TTLServo
        ttl_servo = TTLServo
        print('[arm] TTLServo connected')
    return ttl_servo


def apply_arm_state(prefix, label):
    p = current_params()
    servos = ensure_servos()
    print('[arm] {} order={} dry_run={}'.format(label, tuple(p['safe_order']), p['dry_run_arm']))
    for servo_id in p['safe_order']:
        angle = int(p['{}_s{}'.format(prefix, servo_id)])
        print('[arm] servo={} angle={} speed={}'.format(servo_id, angle, p['arm_speed']))
        if servos is not None:
            servos.servoAngleCtrl(int(servo_id), angle, 1, int(p['arm_speed']))
        time.sleep(float(p['servo_settle_seconds']))


def safe_home():
    apply_arm_state('safe', 'safe_home')


def approaching_arm_state():
    base_stop('Approaching Arm State requires stationary base')
    apply_arm_state('approaching', 'approaching_arm_state')

In [ ]:
def label_for(class_id):
    class_id = int(class_id)
    if 0 <= class_id < len(labels):
        return labels[class_id]
    return 'class_{}'.format(class_id)


def summarize_depth_at(cx_norm, cy_norm, half_size_norm):
    height, width = depth_array.shape[:2]
    x1 = max(0, min(width - 1, int((cx_norm - half_size_norm) * width)))
    x2 = max(x1 + 1, min(width, int((cx_norm + half_size_norm) * width)))
    y1 = max(0, min(height - 1, int((cy_norm - half_size_norm) * height)))
    y2 = max(y1 + 1, min(height, int((cy_norm + half_size_norm) * height)))
    region = depth_array[y1:y2, x1:x2]
    finite = region[np.isfinite(region)]
    if finite.size == 0:
        return {'mean': None, 'min': None, 'max': None, 'count': 0, 'region': (x1, y1, x2, y2)}
    return {
        'mean': float(np.mean(finite)), 'min': float(np.min(finite)),
        'max': float(np.max(finite)), 'count': int(finite.size),
        'region': (x1, y1, x2, y2),
    }


def detection_record(detection):
    class_id = int(detection.ClassID)
    return {
        'class_id': class_id, 'label': label_for(class_id),
        'confidence': float(detection.Confidence),
        'bbox': (float(detection.Left), float(detection.Top), float(detection.Right), float(detection.Bottom)),
    }


def draw_preview(frame, records, observation):
    preview = frame.copy()
    frame_height, frame_width = preview.shape[:2]
    cv2.line(preview, (frame_width // 2, 0), (frame_width // 2, frame_height), (255, 255, 0), 1)
    for record in records:
        left, top, right, bottom = record['bbox']
        is_bottle = record['class_id'] in bottle_class_ids
        color = (0, 255, 0) if is_bottle else (0, 180, 255)
        cv2.rectangle(preview, (int(left), int(top)), (int(right), int(bottom)), color, 2)
        caption = '{} {:.2f}'.format(record['label'], record['confidence'])
        cv2.putText(preview, caption, (int(left), max(15, int(top) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    if observation['bottle'] is not None:
        center = observation['bbox_center']
        cv2.circle(preview, (int(center[0]), int(center[1])), 4, (0, 0, 255), -1)
    ok, encoded = cv2.imencode('.jpg', preview)
    if ok:
        preview_widget.value = encoded.tobytes()


def observe_frame(update_preview=True, confidence_threshold=None):
    load_models()
    cam = start_camera()
    frame = cam.value
    if frame is None:
        raise RuntimeError('camera returned no frame')
    frame = frame.copy()
    if confidence_threshold is None:
        confidence_threshold = float(controls['detection_confidence_k'].value)
    confidence_threshold = float(confidence_threshold)
    detector.SetThreshold(confidence_threshold)

    rgba = cv2.cvtColor(frame, cv2.COLOR_BGR2RGBA)
    detections = detector.Detect(cuda_from_numpy(rgba))
    records = [detection_record(item) for item in detections]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    depth_net.Process(cuda_from_numpy(rgb))
    cuda_sync()

    candidates = [
        record for record in records
        if record['class_id'] in bottle_class_ids and record['confidence'] >= confidence_threshold
    ]
    bottle = max(candidates, key=lambda item: item['confidence']) if candidates else None
    frame_height, frame_width = frame.shape[:2]
    if bottle is None:
        cx, cy = frame_width / 2.0, frame_height / 2.0
    else:
        left, top, right, bottom = bottle['bbox']
        cx, cy = (left + right) / 2.0, (top + bottom) / 2.0
    cx_norm = cx / float(frame_width)
    cy_norm = cy / float(frame_height)
    raw_error_norm = (cx - frame_width / 2.0) / (frame_width / 2.0)
    direction_error_norm = raw_error_norm * int(controls['camera_x_sign'].value)
    depth_stats = summarize_depth_at(
        cx_norm, cy_norm, float(controls['depth_roi_half_size_norm'].value)
    )
    observation = {
        'records': records, 'bottle': bottle, 'bbox_center': (cx, cy),
        'raw_error_norm': raw_error_norm, 'direction_error_norm': direction_error_norm,
        'depth': depth_stats, 'frame_shape': frame.shape, 'confidence_threshold': confidence_threshold,
    }
    if update_preview:
        draw_preview(frame, records, observation)
    return observation


def report_observation(observation, label='frame'):
    bottle = observation['bottle']
    depth = observation['depth']
    if bottle is None:
        print('[{}] bottle=none objects={} center_depth_mean={}'.format(
            label, len(observation['records']),
            'n/a' if depth['mean'] is None else '{:.3f}'.format(depth['mean'])
        ))
        return
    print('[{}] bottle confidence={:.3f} bbox={} center=({:.1f},{:.1f}) error={:+.3f}'.format(
        label, bottle['confidence'], tuple(round(value, 1) for value in bottle['bbox']),
        observation['bbox_center'][0], observation['bbox_center'][1], observation['direction_error_norm']
    ))
    if depth['mean'] is None:
        print('[{}] depth unavailable count=0 region={}'.format(label, depth['region']))
    else:
        print('[{}] depth mean={:.3f} min={:.3f} max={:.3f} count={} region={}'.format(
            label, depth['mean'], depth['min'], depth['max'], depth['count'], depth['region']
        ))


def verify_models_and_camera():
    print('[verify] models and camera verification start')
    load_models()
    start_camera()
    first = observe_frame(update_preview=True)
    report_observation(first, 'verify-1')
    time.sleep(0.2)
    second = observe_frame(update_preview=True)
    report_observation(second, 'verify-2')
    print('[verify] success: two camera frames processed by both models')
    return second


def predict_one_frame():
    observation = observe_frame(update_preview=True)
    report_observation(observation, 'single')
    return observation

In [ ]:
def observation_pause():
    seconds = float(controls['observation_pause_seconds'].value)
    if seconds > 0:
        time.sleep(seconds)


def search_state():
    p = current_params()
    threshold = float(p['search_confidence_k'])
    print('[search] start confidence={:.2f} direction={} speed={:.2f} max_seconds={:.2f}'.format(
        threshold, p['scan_direction'], p['search_turn_speed'], p['search_max_seconds']
    ))
    initial = observe_frame(update_preview=True, confidence_threshold=threshold)
    report_observation(initial, 'search-initial')
    if initial['bottle'] is not None:
        base_stop('bottle already visible before Search rotation')
        print('[search] bottle already visible; skip rotation')
        return initial

    timer = start_continuous_search_turn(p['scan_direction'], p['search_turn_speed'], p['search_max_seconds'])
    started = time.time()
    frame_index = 0
    try:
        while time.time() - started < p['search_max_seconds']:
            observation = observe_frame(update_preview=True, confidence_threshold=threshold)
            frame_index += 1
            report_observation(observation, 'search-{}'.format(frame_index))
            if observation['bottle'] is not None:
                print('[search] bottle acquired after {:.2f}s'.format(time.time() - started))
                return observation
            observation_pause()
    finally:
        if timer is not None:
            timer.cancel()
        base_stop('Search State cleanup')
    raise RuntimeError('no bottle above Search confidence before maximum rotation time')


def align_to_bottle(initial_observation=None):
    p = current_params()
    observation = initial_observation
    for step in range(1, int(p['align_max_steps']) + 1):
        if observation is None:
            observation = observe_frame(update_preview=True)
        if observation['bottle'] is None:
            base_stop('bottle lost during alignment')
            raise RuntimeError('bottle lost during bbox alignment')
        error = float(observation['direction_error_norm'])
        report_observation(observation, 'align-{}'.format(step))
        if abs(error) <= p['align_tolerance_norm']:
            base_stop('bbox centered')
            print('[align] centered error={:+.3f} tolerance={:.3f}'.format(error, p['align_tolerance_norm']))
            return observation
        direction = 'right' if error > 0 else 'left'
        base_pulse(direction, p['align_turn_speed'], p['align_pulse_seconds'], 'bbox alignment')
        observation_pause()
        observation = None
    raise RuntimeError('bbox alignment exceeded align_max_steps')


def approach_until_depth(initial_observation=None):
    p = current_params()
    commanded_forward_seconds = 0.0
    observation = initial_observation
    step = 0
    while commanded_forward_seconds <= p['approach_max_seconds']:
        step += 1
        if observation is None:
            observation = observe_frame(update_preview=True)
        if observation['bottle'] is None:
            base_stop('bottle lost during approach')
            raise RuntimeError('bottle lost during depth approach')
        if abs(observation['direction_error_norm']) > p['align_tolerance_norm']:
            print('[approach] bbox drifted; realigning before forward movement')
            observation = align_to_bottle(observation)
        report_observation(observation, 'approach-{}'.format(step))
        mean_depth = observation['depth']['mean']
        if mean_depth is None:
            raise RuntimeError('DepthNet returned no finite values in bottle ROI')
        if mean_depth < p['stop_depth_m']:
            base_stop('depth threshold reached')
            print('[approach] success mean_depth={:.3f} < m={:.3f}'.format(mean_depth, p['stop_depth_m']))
            return observation
        if commanded_forward_seconds >= p['approach_max_seconds']:
            break
        pulse = min(p['approach_pulse_seconds'], p['approach_max_seconds'] - commanded_forward_seconds)
        base_pulse('forward', p['approach_speed'], pulse, 'depth approach')
        commanded_forward_seconds += pulse
        observation_pause()
        observation = None
    raise RuntimeError('approach exceeded maximum commanded movement time')


def run_scan_align_approach_demo():
    p = current_params()
    print('[flow] demo start dry_run_base={} dry_run_arm={} k={:.2f} m={:.3f}'.format(
        p['dry_run_base'], p['dry_run_arm'], p['detection_confidence_k'], p['stop_depth_m']
    ))
    try:
        load_models()
        start_camera()
        safe_home()
        base_stop('before scan')
        found = search_state()
        aligned = align_to_bottle(found)
        approaching_arm_state()
        realigned = align_to_bottle(None)
        result = approach_until_depth(realigned)
        print('[flow] demo success')
        return result
    finally:
        base_stop('demo finally')

In [ ]:
save_button = widgets.Button(description='Save Params', button_style='info')
load_button = widgets.Button(description='Load Models', button_style='primary')
verify_button = widgets.Button(description='Start Camera + Verify', button_style='info')
predict_button = widgets.Button(description='Predict One Frame')
camera_stop_button = widgets.Button(description='Stop Camera', button_style='warning')
safe_button = widgets.Button(description='Safe Home', button_style='warning')
search_button = widgets.Button(description='Search State')
approaching_button = widgets.Button(description='Approaching Arm State')
run_button = widgets.Button(description='Run Demo', button_style='success')
base_stop_button = widgets.Button(description='Stop Base', button_style='danger')
clear_button = widgets.Button(description='Clear Log')
preview_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='520px'))
log_output = widgets.Output(layout={
    'border': '1px solid #bbb', 'height': '360px', 'overflow_y': 'auto', 'width': '100%'
})
busy = False
action_buttons = [save_button, load_button, verify_button, predict_button, safe_button, search_button, approaching_button, run_button]


def run_with_log(function):
    def wrapped(_=None):
        global busy
        with log_output:
            if busy:
                print('[busy] another operation is running')
                return
            busy = True
            for button in action_buttons:
                button.disabled = True
            try:
                function()
            except Exception as exc:
                print('[error] {}: {}'.format(type(exc).__name__, exc))
                print('[hint] TensorRT logs may appear in the Jupyter server terminal.')
            finally:
                busy = False
                for button in action_buttons:
                    button.disabled = False
    return wrapped


save_button.on_click(run_with_log(save_params))
load_button.on_click(run_with_log(load_models))
verify_button.on_click(run_with_log(verify_models_and_camera))
predict_button.on_click(run_with_log(predict_one_frame))
camera_stop_button.on_click(run_with_log(stop_camera))
safe_button.on_click(run_with_log(safe_home))
search_button.on_click(run_with_log(search_state))
approaching_button.on_click(run_with_log(approaching_arm_state))
run_button.on_click(run_with_log(run_scan_align_approach_demo))
base_stop_button.on_click(run_with_log(lambda: base_stop('UI stop button')))
clear_button.on_click(lambda _: log_output.clear_output())

button_row_1 = widgets.HBox([save_button, load_button, verify_button, predict_button, camera_stop_button])
button_row_2 = widgets.HBox([safe_button, search_button, approaching_button, run_button, base_stop_button, clear_button])
ui = widgets.VBox([
    button_row_1, button_row_2,
    preview_widget,
    widgets.HTML('<b>Log output</b>'), log_output,
    widgets.HTML('<b>Safety</b>'), widgets.HBox([controls['dry_run_base'], controls['dry_run_arm']]),
    widgets.HTML('<b>Detection and depth</b>'),
    controls['detection_confidence_k'], controls['stop_depth_m'], controls['depth_roi_half_size_norm'], controls['camera_x_sign'],
    widgets.HTML('<b>Search State - continuous capture while rotating</b>'),
    controls['search_confidence_k'], controls['scan_direction'], controls['search_turn_speed'], controls['search_max_seconds'],
    widgets.HTML('<b>BBox center alignment</b>'),
    controls['align_turn_speed'], controls['align_pulse_seconds'], controls['align_tolerance_norm'], controls['align_max_steps'],
    widgets.HTML('<b>Depth approach</b>'),
    controls['approach_speed'], controls['approach_pulse_seconds'], controls['approach_max_seconds'], controls['observation_pause_seconds'],
    widgets.HTML('<b>Tunable camera Safe Home</b>'), safe_order_widget,
    controls['safe_s1'], controls['safe_s2'], controls['safe_s3'], controls['safe_s4'], controls['safe_s5'],
    widgets.HTML('<b>Approaching Arm State - base remains stopped</b>'),
    controls['approaching_s1'], controls['approaching_s2'], controls['approaching_s3'], controls['approaching_s4'], controls['approaching_s5'],
    controls['arm_speed'], controls['servo_settle_seconds'],
])
display(ui)

## Test order

1. Run all cells from top to bottom.
2. Keep both dry-run boxes enabled. Click **Load Models**, then **Start Camera + Verify**. Confirm two changing DepthNet reports and plausible SSD detections.
3. Use **Predict One Frame** with and without a bottle. Verify confidence, bbox center, signed horizontal error, and bottle-centered depth values.
4. Tune and test **Safe Home** so the camera faces forward without arm interference.
5. Calibrate `camera_x_sign`: when a bottle appears on the right, `error` should be positive. Select `mirrored` if it is reversed.
6. Test **Search State** separately. It checks one stationary frame first, then continuously captures while rotating until `search_confidence_k` is reached or `search_max_seconds` expires.
7. Tune **Approaching Arm State** separately with the base stopped. The full demo realigns the bbox after this arm movement.
8. Tune alignment with short pulses, then tune approach speed and `stop_depth_m` from observed raw DepthNet values.
9. Save parameters, disable only the hardware dry-run boxes you are ready to test, and run the demo with space around the robot.
10. Use **Stop Camera** after testing. Search has a timer stop, every chassis pulse has a `finally` stop, and the full demo also stops the base in `finally`.